## Step 1: Load PDF documents

In [ ]:
# Step 1: Load every PDF from ../data/pdf into LangChain Document objects (one per page)
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

## Step 2: Split documents into chunks

In [ ]:
# Step 2: Split the documents into overlapping chunks so each fits an embedding and retrieves precisely
from typing import List, Any

from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(
    documents: List[Any],
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Any]:
    """
    Split documents into smaller chunks for better RAG performance.

    Args:
        documents: List of documents to split
        chunk_size: Maximum number of characters in each chunk
        chunk_overlap: Number of characters shared between chunks

    Returns:
        List of chunked documents
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(
        f"Split {len(documents)} documents "
        f"into {len(split_docs)} chunks"
    )

    # Show an example chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


chunks = split_documents(pdf_documents)
chunks

## Imports for embeddings and vector store

In [ ]:
# Imports for embeddings, the Chroma vector store and similarity calculations
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

## Step 3: Embedding manager

In [ ]:
# Step 3: Embedding manager - wraps a SentenceTransformer model that turns text into 384-dim vectors
from typing import List

import numpy as np
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager"""

        self.model_name = model_name
        self.model = None

        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed

        Returns:
            NumPy array of embeddings with shape
            (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

## Step 4: Vector store (ChromaDB)

In [ ]:
# Step 4: Vector store - persists chunk text, metadata and embeddings in a ChromaDB collection
import os
import uuid
from typing import List, Any

import chromadb
import numpy as np


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """Initialize the vector store"""

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent directory if it doesn't exist
            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Add documents and their embeddings to the vector store"""

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document text
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise
        
vector_store = VectorStore()
vector_store



## Step 5: Generate embeddings and store them in the vector store

In [ ]:
# Step 5: Embed all chunks and store them in the vector store
# (re-running this adds the chunks again, creating duplicates)
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)


## Preview chunk texts

In [ ]:
# Preview the chunk texts that will be embedded
texts=[doc.page_content for doc in chunks]
texts

## Step 6: Retriever (query -> relevant chunks)

In [ ]:
# Step 6: Retriever - embeds the query and fetches the most similar chunks from the vector store
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents
            and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    # (approximation: Chroma's default metric is L2)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

## Test the retriever

In [ ]:
# Test the retriever: fetch the top matching chunks for a sample query
rag_retriever.retrieve("what is the acme ai is?")

## Step 7: Set up the Groq LLM

In [ ]:
# Step 7: Set up the Groq LLM (API key is read from the .env file)
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Loads GROQ_API_KEY from the .env file in the project root
load_dotenv(dotenv_path="../.env")

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing. Add it to the .env file.")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)


## Step 8: RAG function (retrieve -> context -> answer)

In [ ]:
# Step 8: RAG function - retrieve chunks, build a prompt with that context, and ask the LLM
from langchain_core.prompts import PromptTemplate


def rag_simple(
    query: str,
    retriever: RAGRetriever,
    llm: ChatGroq,
    top_k: int = 3
):
    """
    Simple RAG: retrieve relevant chunks, put them in the prompt,
    and let the Groq LLM answer using only that context.

    Returns:
        The answer string
    """

    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc["content"] for doc in results)

    if not context:
        return "No relevant context found."

    prompt = PromptTemplate(
        template="""Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:""",
        input_variables=["context", "query"]
    )

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


## Run the RAG pipeline

In [ ]:
# Ask a question through the full RAG pipeline and print the answer
answer = rag_simple("What is acme ai?", rag_retriever, llm)
print(answer)


## Enhanced RAG pipeline (with sources)

In [ ]:
# Enhanced RAG function: same retrieve -> context -> answer logic, but also returns sources, confidence and context
def rag_with_sources(
    query: str,
    retriever: RAGRetriever,
    llm: ChatGroq,
    top_k: int = 3,
    min_score: float = 0.0,
    return_context: bool = False
):
    """
    Same pipeline as rag_simple, but returns the answer together with
    the source chunks used, a confidence score and (optionally) the context.

    Returns:
        Dict with the answer, sources, confidence (0 to 1) and optional context
    """

    # Retrieve the most relevant chunks (ignoring those below min_score)
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {"answer": "No relevant context found.", "sources": [], "confidence": 0.0, "context": ""}

    # Prepare context and sources
    context = "\n\n".join([doc["content"] for doc in results])
    sources = [
        {
            "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
            "page": doc["metadata"].get("page", "unknown"),
            "score": round(doc["similarity_score"], 3),
            "preview": doc["content"][:300] + "...",
        }
        for doc in results
    ]

    # Confidence = similarity of the best matching chunk, kept between 0 and 1
    confidence = round(max(0.0, min(1.0, max([doc["similarity_score"] for doc in results]))), 3)

    # Generate answer
    prompt = PromptTemplate(
        template="""Use the following context to answer the question concisely.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:""",
        input_variables=["context", "query"]
    )
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }
    if return_context:
        output["context"] = context
    return output


## Run the enhanced RAG pipeline

In [ ]:
# Example usage:
result = rag_with_sources("What is acme ai?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])
print("Context Preview:", result["context"][:300])
